# PCA Dynamic NWB-Only (v2)

NWB-only pipeline. No `df_units.json` or `df_stim.json` is needed.


### Suggestions on how to improve Notebook
1. okay how can i now run each of these PCAs on a per probe level. For example in the df_units there is the column header "probe", values for "probe" can be either (A,B,C,D,E,F). 

### Imports


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pynwb import NWBHDF5IO
from scipy.ndimage import gaussian_filter1d

sns.set_context("talk")
sns.set_style("white")
warnings.filterwarnings("ignore")


### NWB Loader


In [2]:
class nwb_loader:
    def __init__(self, nwb_path):
        self.nwb_path = str(nwb_path)
        self.io = None
        self.nwb = None
        self.load_nwb()

    def load_nwb(self):
        self.io = NWBHDF5IO(self.nwb_path, "r", load_namespaces=True)
        self.nwb = self.io.read()
        return self.nwb

    def trials(self):
        return self.nwb.trials.to_dataframe() if self.nwb.trials is not None else pd.DataFrame()

    def units(self):
        return self.nwb.units.to_dataframe() if self.nwb.units is not None else pd.DataFrame()

    def optogenetics_states(self):
        if "optogenetics_states" in self.nwb.intervals:
            return self.nwb.intervals["optogenetics_states"].to_dataframe()
        return pd.DataFrame()

    def close(self):
        try:
            if self.io is not None:
                self.io.close()
        except Exception:
            pass


### Config


In [3]:
NWB_PATH = Path(r"G:\Grant\neuropixels\nwb\Reach15")

WINDOW_START_S = -0.5
WINDOW_END_S = 3.5
BIN_SIZE_S = 0.05
N_COMPONENTS = 12
SMOOTH_SIGMA = 2

LABEL_MODE = "block"   # 'block' or 'stimulus'
EVENT_INCLUDE = None    # e.g., ['reach'] or None

# Memory safety controls
MAX_EVENTS = 5000                 # hard cap on analyzed events (None disables)
EVENT_SUBSAMPLE_MODE = "uniform"  # 'uniform' or 'first'
MAX_TENSOR_GB = 8.0               # abort if trials tensor estimate exceeds this

PROBE_FILTER = None     # 'A'..'F' or None
KSLABEL_FILTER = None   # 'good'/'mua' or 2/1 or None
UNIT_FILTER_QUERY = None
MAX_UNITS = None

OUT_DIR = Path.cwd() / "extra_files"
OUT_DIR.mkdir(parents=True, exist_ok=True)


### Helpers


In [27]:
def resolve_nwb_path(p: Path) -> Path:
    if not p.exists():
        raise FileNotFoundError(f"NWB path not found: {p}")
    if p.is_file():
        return p
    nwb_files = sorted(list(p.rglob("*.nwb")))
    if len(nwb_files) == 1:
        return nwb_files[0]
    if len(nwb_files) > 1:
        raise ValueError("Multiple NWB files found. Point NWB_PATH to one file.")
    files = [x for x in p.iterdir() if x.is_file()]
    if len(files) == 1:
        return files[0]
    raise ValueError("Could not resolve a unique NWB file path.")


def zscore_rows(x: np.ndarray) -> np.ndarray:
    ss = StandardScaler(with_mean=True, with_std=True)
    return ss.fit_transform(x.T).T


def choose_event_time_col(df: pd.DataFrame) -> str:
    for c in ['stimulus']:
        if c in df.columns:
            return c
    raise ValueError(f"No event-time column found in trials table. Columns: {list(df.columns)}")


def normalize_kslabel(v):
    if pd.isna(v):
        return np.nan
    try:
        iv = int(float(v))
        if iv == 2:
            return "good"
        if iv == 1:
            return "mua"
    except Exception:
        pass
    s = str(v).strip().lower()
    if s in {"2", "good", "single", "singleunit", "single_unit"}:
        return "good"
    if s in {"1", "mua", "multi", "multiunit", "multi_unit"}:
        return "mua"
    return s


def infer_is_opto(df_trials: pd.DataFrame) -> pd.Series:
    if "optogenetics_LED_state" in df_trials.columns:
        c = df_trials["optogenetics_LED_state"]
        if pd.api.types.is_numeric_dtype(c):
            return pd.to_numeric(c, errors="coerce").fillna(0) > 0
        return c.astype(str).str.lower().isin(["1", "true", "on", "high"])
    if "stimulus" in df_trials.columns:
        s = df_trials["stimulus"].astype(str).str.lower()
        return s.str.contains("opto|laser|led|stim", regex=True)
    return pd.Series([False] * len(df_trials), index=df_trials.index)


def build_block_labels(is_opto: pd.Series) -> pd.DataFrame:
    is_opto = is_opto.astype(bool).reset_index(drop=True)
    block_id = (is_opto != is_opto.shift(1, fill_value=is_opto.iloc[0])).cumsum()

    mapping = {}
    seen_opto = False
    opto_k = 1
    wash_k = 1
    for b in block_id.unique():
        state = bool(is_opto[block_id == b].iloc[0])
        if state:
            mapping[b] = f"opto_epoch_{opto_k}"
            opto_k += 1
            seen_opto = True
        else:
            mapping[b] = "baseline" if not seen_opto else f"washout_epoch_{wash_k}"
            if seen_opto:
                wash_k += 1

    return pd.DataFrame({"block_id": block_id, "is_opto": is_opto, "block_label": block_id.map(mapping)})


def find_probe_col(df: pd.DataFrame):
    for c in ["probe", "probe_name", "probe_id", "probe_letter", "electrode_group"]:
        if c in df.columns:
            return c
    return None


def find_kslabel_col(df: pd.DataFrame):
    for c in ["KSLabel", "kslabel", "ks_label", "label", "quality"]:
        if c in df.columns:
            return c
    return None


def pick_region_col(df: pd.DataFrame):
    for c in ["brain_region", "location", "region", "acronym", "structure", "ccf_acronym"]:
        if c in df.columns:
            return c
    return None


### Load NWB Tables


In [28]:
resolved_nwb = resolve_nwb_path(NWB_PATH)
mouse = nwb_loader(resolved_nwb)

df_trials = mouse.trials().reset_index(drop=True)
df_units = mouse.units().reset_index(drop=True)

print("NWB:", resolved_nwb)
print("trials shape:", df_trials.shape)
print("units shape:", df_units.shape)
print("trials columns:", list(df_trials.columns))
print("units columns:", list(df_units.columns))


NWB: G:\Grant\neuropixels\nwb\Reach15
trials shape: (1352824, 4)
units shape: (4600, 9)
trials columns: ['start_time', 'stop_time', 'stimulus', 'optogenetics_LED_state']
units columns: ['depth', 'xpos', 'ypos', 'label', 'KSlabel', 'KSamplitude', 'KScontamination', 'probe', 'spike_times']


In [29]:
# print unique names inside stimulus
unit_probes = np.array(df_units.probe.unique())
unit_headers = df_units.columns.tolist()
df_units_dic = {}

print('===== Total units Per Probe ====')
for probe in unit_probes:
    df = df_units[df_units['probe']==probe]
    print(f'{probe}: ', len(df))
    df_units_dic[probe] = df
print('\n')
probe_letter = 'A'
print(f'Comapre with Above: total units in probe {probe_letter}: ', len(df_units_dic[probe_letter]))
df_units_dic[probe_letter][0:5]

===== Total units Per Probe ====
A:  554
B:  1316
C:  355
D:  820
E:  864
F:  691


Comapre with Above: total units in probe A:  554


,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,spike_times
0,3120.0,277.0,30.0,0,2,9.5,14.5,A,"[9938.427594575005, 9938.443329123113, 9938.44..."
1,3120.0,277.0,30.0,0,2,10.3,17.2,A,"[9938.363989665882, 9938.408826460065, 9938.42..."
2,3120.0,59.0,30.0,0,1,9.0,0.0,A,"[9938.135738714089, 9939.27045963741, 9940.801..."
3,2775.0,27.0,375.0,0,2,8.5,0.0,A,"[9938.897430842511, 9940.381573391209, 9941.04..."
4,2745.0,59.0,405.0,0,2,9.1,7.4,A,"[9938.29451763671, 9938.519134975282, 9938.609..."


### Load bombcell Results

In [30]:
qm_df_list = []
qm_dic = {}
# set root bombcell path 
for probe in unit_probes:
    print(f'Loading Probe {probe} quality_metrics.csv')
    bc_qm_path = fr"H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260222_1618\kilosort4_{probe}\bombcell\Probe_{probe}_quality_metrics.csv"
    # print('      ',bc_qm_path)
    qm_df = pd.read_csv(bc_qm_path)
    qm_df_list.append(qm_df)
    qm_dic[probe] = qm_df

### Look at one of the probes quality_metrics.csv from the dic
probe_letter = "B"
qm_dic[probe_letter]

Loading Probe A quality_metrics.csv
Loading Probe B quality_metrics.csv
Loading Probe C quality_metrics.csv
Loading Probe D quality_metrics.csv
Loading Probe E quality_metrics.csv
Loading Probe F quality_metrics.csv


,Bombcell_unit_type,bc_roiLabel,cluster_id,phy_clusterID,nSpikes,nPeaks,nTroughs,waveformDuration_peakTrough,spatialDecaySlope,waveformBaselineFlatness,...,maxDriftEstimate,cumDriftEstimate,rawAmplitude,signalToNoiseRatio,isolationDistance,Lratio,silhouetteScore,useTheseTimesStart,useTheseTimesStop,maxChannels
0,MUA,IN_ROI,0,0,342314.0,1.0,1.0,166.666667,0.013970,0.020126,...,11.119194,57.314980,72.875717,14.510937,NaN,NaN,NaN,0.000067,9295.179267,2
1,MUA,IN_ROI,1,1,93291.0,2.0,1.0,133.333333,0.021148,0.050982,...,25.348637,271.297123,23.371343,2.061966,NaN,NaN,NaN,0.000067,9295.179267,0
2,NOISE,IN_ROI,2,2,21451.0,1.0,1.0,700.000000,0.017437,0.126556,...,9.852627,236.820278,51.590074,13.997928,NaN,NaN,NaN,0.000067,9295.179267,1
3,MUA,IN_ROI,3,3,48982.0,1.0,1.0,200.000000,0.020088,0.015577,...,21.799164,422.647194,65.607953,8.144144,NaN,NaN,NaN,0.000067,9295.179267,1
4,GOOD,IN_ROI,4,4,17269.0,1.0,1.0,233.333333,0.030410,0.021121,...,29.794754,330.584137,47.304186,7.611712,NaN,NaN,NaN,0.000067,9295.179267,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1311,MUA,OUT_ROI,1311,1311,94232.0,1.0,1.0,133.333333,0.018073,0.024039,...,3778.352539,33845.103516,20.537167,2.396758,NaN,NaN,NaN,0.000067,9295.179267,357
1312,MUA,OUT_ROI,1312,1312,1486.0,1.0,1.0,300.000000,0.025199,0.036436,...,25.231201,118.135742,13.312992,3.542415,NaN,NaN,NaN,0.000067,9295.179267,357
1313,NON-SOMA,OUT_ROI,1313,1313,1175.0,1.0,1.0,233.333333,0.022037,0.047541,...,23.249023,273.681641,46.155959,5.259838,NaN,NaN,NaN,0.000067,9295.179267,357
1314,MUA,OUT_ROI,1314,1314,2601.0,1.0,1.0,166.666667,0.024749,0.053390,...,20.525635,31.273438,11.809050,2.343361,NaN,NaN,NaN,0.000067,9295.179267,357


In [31]:
### Look at one of the probes quality_metrics.csv from the dic
probe_letter = "E"
qm_dic[probe_letter]

,Bombcell_unit_type,bc_roiLabel,cluster_id,phy_clusterID,nSpikes,nPeaks,nTroughs,waveformDuration_peakTrough,spatialDecaySlope,waveformBaselineFlatness,...,maxDriftEstimate,cumDriftEstimate,rawAmplitude,signalToNoiseRatio,isolationDistance,Lratio,silhouetteScore,useTheseTimesStart,useTheseTimesStop,maxChannels
0,NON-SOMA,IN_ROI,0,0,302.0,1.0,1.0,200.0,0.018677,0.010223,...,24.241371,469.517036,17.205716,3.098545,NaN,NaN,NaN,0.0003,9295.2395,25
1,NON-SOMA,IN_ROI,1,1,23063.0,1.0,1.0,200.0,0.023395,0.034017,...,19.156597,157.544125,47.435221,14.550487,NaN,NaN,NaN,0.0003,9295.2395,1
2,NOISE,IN_ROI,2,2,41013.0,2.0,2.0,400.0,0.037405,0.127608,...,13.474518,113.402157,68.206922,12.086486,NaN,NaN,NaN,0.0003,9295.2395,21
3,MUA,IN_ROI,3,3,2558.0,2.0,1.0,500.0,0.023233,0.142165,...,42.070847,1042.221340,82.300007,15.866048,NaN,NaN,NaN,0.0003,9295.2395,21
4,NOISE,IN_ROI,4,4,25540.0,2.0,2.0,100.0,0.055958,0.036925,...,37.151985,558.720367,10.219774,1.739762,NaN,NaN,NaN,0.0003,9295.2395,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,MUA,OUT_ROI,859,859,39987.0,2.0,1.0,700.0,0.069340,0.161780,...,53.915283,662.756836,151.221590,63.154808,NaN,NaN,NaN,0.0003,9295.2395,380
860,MUA,OUT_ROI,860,860,54200.0,2.0,1.0,200.0,0.060062,0.009964,...,48.777100,758.604248,205.234832,113.377360,NaN,NaN,NaN,0.0003,9295.2395,380
861,MUA,OUT_ROI,861,861,200691.0,2.0,1.0,600.0,0.020898,0.113920,...,63.108643,508.811035,24.369253,2.723692,NaN,NaN,NaN,0.0003,9295.2395,357
862,MUA,OUT_ROI,862,862,79925.0,2.0,1.0,600.0,0.019167,0.127718,...,24.424072,342.160156,32.999388,2.973003,NaN,NaN,NaN,0.0003,9295.2395,357


### Verify the bombcell quality metrics dictonary matches the number of untis in df_units on a per probe basis

In [32]:
print('===== Total units Per Probe ====')
for probe in unit_probes:
    df = df_units[df_units['probe']==probe]
    df_units_len = len(df)
    qm_dic_len = len(qm_dic[probe])
    if df_units_len == qm_dic_len:
        print(f'df_units {probe}: ', df_units_len)
        print(f'qm_dic_len {probe}: ', qm_dic_len)
        print('------------------------')
    else:
        print(f'**qualitt_metrics.csv and df_units for Probe {probe} do NOT have same number of units***')
        print(f'df_units {probe}: ', df_units_len)
        print(f'qm_dic_len {probe}: ', qm_dic_len)
        print('------------------------')


===== Total units Per Probe ====
df_units A:  554
qm_dic_len A:  554
------------------------
df_units B:  1316
qm_dic_len B:  1316
------------------------
df_units C:  355
qm_dic_len C:  355
------------------------
df_units D:  820
qm_dic_len D:  820
------------------------
df_units E:  864
qm_dic_len E:  864
------------------------
df_units F:  691
qm_dic_len F:  691
------------------------


### How to Run Bombcell if NOT done
- Step 1: Use BOMBCELL github repo
- Step 2: Run the following file -> C:\Users\user\Documents\github\bombcell\py_bombcell\grant\running_BC\BC_open_ephys_unified.ipynb
- Step 3: Run the following file -> C:\Users\user\Documents\github\bombcell\py_bombcell\grant\phy_update\add_brain_region_to_phy.ipynb
- Doing this should produce the quality_metrics.csv you need 

In [33]:
import pandas as pd
clusster_bc_dic = {}
print('===== Total units Per Probe ====')
for probe in unit_probes:
    file_path = fr"H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260222_1618\kilosort4_{probe}\cluster_bc_classificationReason.tsv"
    df1 = pd.read_csv(file_path, sep='\t')
    df2 = df_units[df_units['probe']==probe]
    df1_len = len(df1)
    df2_len = len(df2)
    if df1_len == df2_len:
        clusster_bc_dic[probe] = df1
        print(f'SUCCESS probe {probe}')
        print('------------------------')
    else:
        print(f"ERROR PROBE {probe}")
        print(f'**qualitt_metrics.csv and df_units for Probe {probe} do NOT have same number of units***')
        print(f'cluster.tsv {probe}: ', df1_len)
        print(f'df_units {probe}: ', df2_len)
        print('------------------------')


===== Total units Per Probe ====
SUCCESS probe A
------------------------
SUCCESS probe B
------------------------
SUCCESS probe C
------------------------
SUCCESS probe D
------------------------
SUCCESS probe E
------------------------
SUCCESS probe F
------------------------


### Verify cluster_bc_classificationReason.tsv dic is correct per probe

In [34]:
probe_letter = 'A'
clusster_bc_dic[probe_letter][0:3]


,cluster_id,bc_classificationReason,bc_ROI,Brain_Region
0,0,NOISE,IN_ROI,SIM
1,1,NOISE,IN_ROI,SIM
2,2,NOISE,IN_ROI,SIM


### Combine Bombcell quality_metrics.csv and cluster_bc_classificationReason metrics columns with df_units 

In [35]:
merged_dic = {}

for probe in unit_probes:
    u = df_units_dic[probe].copy().reset_index(drop=True)
    qm = qm_dic[probe].copy().reset_index(drop=True)
    cl = clusster_bc_dic[probe].copy().reset_index(drop=True)

    # Ensure cluster_id exists and is same dtype
    if "cluster_id" not in u.columns:
        # if needed, bring from qm by position first
        u["cluster_id"] = qm["cluster_id"].values

    u["cluster_id"] = pd.to_numeric(u["cluster_id"], errors="coerce")
    qm["cluster_id"] = pd.to_numeric(qm["cluster_id"], errors="coerce")
    cl["cluster_id"] = pd.to_numeric(cl["cluster_id"], errors="coerce")

    # Merge by key (safe), not by index alignment
    m = (
        u.merge(
            qm[["cluster_id", "nSpikes", "maxDriftEstimate", "maxChannels"]],
            on="cluster_id",
            how="left",
        )
        .merge(
            cl[["cluster_id", "bc_classificationReason", "bc_ROI", "Brain_Region"]],
            on="cluster_id",
            how="left",
        )
        .rename(
            columns={
                "bc_classificationReason": "bc_label",
                "bc_ROI": "in_brainRegion",
                "Brain_Region": "brain_region",
            }
        )
    )

    merged_dic[probe] = m

    print(
        probe,
        "units=", len(m),
        "in_brainRegion non-null=", m["in_brainRegion"].notna().sum()
    )


A units= 554 in_brainRegion non-null= 554
B units= 1316 in_brainRegion non-null= 1316
C units= 355 in_brainRegion non-null= 355
D units= 820 in_brainRegion non-null= 820
E units= 864 in_brainRegion non-null= 864
F units= 691 in_brainRegion non-null= 691


### Verify the new merged_dic has data from quality_metrics.csv and values from df_units

In [36]:
probe_letter = 'A'
merged_dic[probe_letter][0:3]


,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,spike_times,cluster_id,nSpikes,maxDriftEstimate,maxChannels,bc_label,in_brainRegion,brain_region
0,3120.0,277.0,30.0,0,2,9.5,14.5,A,"[9938.427594575005, 9938.443329123113, 9938.44...",0,14579.0,29.717560,52,NOISE,IN_ROI,SIM
1,3120.0,277.0,30.0,0,2,10.3,17.2,A,"[9938.363989665882, 9938.408826460065, 9938.42...",1,35469.0,93.164818,52,NOISE,IN_ROI,SIM
2,3120.0,59.0,30.0,0,1,9.0,0.0,A,"[9938.135738714089, 9939.27045963741, 9940.801...",2,176.0,4.297394,5,NOISE,IN_ROI,SIM


In [37]:
probe_letter = 'B'
merged_dic[probe_letter][0:3]

,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,spike_times,cluster_id,nSpikes,maxDriftEstimate,maxChannels,bc_label,in_brainRegion,brain_region
0,6960.0,59.0,40.0,1,1,12.3,89.3,B,"[9938.120278208322, 9938.182216212743, 9938.21...",0,342314.0,11.119194,2,MUA,OUT_ROI,NaN
1,6980.0,43.0,20.0,2,2,11.5,5.1,B,"[9938.444535996017, 9939.917013711796, 9958.09...",1,93291.0,25.348637,0,MUA,OUT_ROI,NaN
2,6980.0,11.0,20.0,0,1,9.5,43.4,B,"[9939.385306945673, 9940.727441498186, 9957.55...",2,21451.0,9.852627,1,NOISE,OUT_ROI,NaN


In [38]:
probe_letter = 'C'
merged_dic[probe_letter][0:3]


,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,spike_times,cluster_id,nSpikes,maxDriftEstimate,maxChannels,bc_label,in_brainRegion,brain_region
0,1635.0,59.0,15.0,0,2,9.8,13.0,C,"[9938.799410308768, 9940.716007243334, 9941.67...",0,23708.0,28.165009,3,MUA,IN_ROI,MoP
1,1530.0,277.0,120.0,0,1,9.1,54.9,C,"[9941.521003949643, 9941.812726914572, 9943.79...",1,3619.0,38.575857,64,NON-SOMA,IN_ROI,MoP
2,1530.0,277.0,120.0,0,2,12.6,18.0,C,"[9938.995592419455, 9939.525680201354, 9941.51...",2,20943.0,71.238548,64,MUA,IN_ROI,MoP


In [39]:
probe_letter = 'D'
merged_dic[probe_letter][0:3]

,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,spike_times,cluster_id,nSpikes,maxDriftEstimate,maxChannels,bc_label,in_brainRegion,brain_region
0,4150.0,59.0,450.0,0,1,9.2,118.7,D,"[9939.54484716702, 9939.950746703747, 9948.824...",0,5522.0,5.877594,109,NON-SOMA,IN_ROI,VaL
1,4135.0,27.0,465.0,2,2,11.7,0.0,D,"[9939.698126246403, 9953.110221524967, 9965.41...",1,1062.0,16.367188,110,NON-SOMA,IN_ROI,VaL
2,4150.0,27.0,450.0,0,1,9.3,195.6,D,"[9939.665556942206, 9946.630463448884, 9949.59...",2,4015.0,7.174561,108,NOISE,IN_ROI,VaL


In [40]:
probe_letter = 'E'
merged_dic[probe_letter][0:3]

,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,spike_times,cluster_id,nSpikes,maxDriftEstimate,maxChannels,bc_label,in_brainRegion,brain_region
0,5740.0,11.0,260.0,0,1,8.9,253.7,E,"[9939.79351277194, 9946.915350412626, 9965.995...",0,302.0,24.241371,25,NON-SOMA,OUT_ROI,NaN
1,5980.0,11.0,20.0,0,1,9.4,224.0,E,"[10029.529560965859, 10029.536061407536, 10029...",1,23063.0,19.156597,1,NON-SOMA,OUT_ROI,NaN
2,5780.0,11.0,220.0,0,1,12.7,64.4,E,"[9991.106788657915, 10157.405349147633, 10159....",2,41013.0,13.474518,21,NOISE,OUT_ROI,NaN


In [41]:
probe_letter = 'F'
merged_dic[probe_letter][0:3]

,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,spike_times,cluster_id,nSpikes,maxDriftEstimate,maxChannels,bc_label,in_brainRegion,brain_region
0,4580.0,43.0,20.0,2,2,10.0,3.3,F,"[9939.864104353275, 9976.476037177195, 9977.54...",0,55328.0,19.892956,0,NON-SOMA,OUT_ROI,NaN
1,4560.0,59.0,40.0,0,1,9.3,88.6,F,"[9938.094331608718, 9938.194672846303, 9938.28...",1,62990.0,9.715141,2,MUA,OUT_ROI,NaN
2,4580.0,11.0,20.0,0,2,9.3,18.3,F,"[9963.663282327045, 10022.517000744512, 10036....",2,15063.0,18.795841,1,NON-SOMA,OUT_ROI,NaN


### PCA Extension: Per Probe + ROI + Event Selection

Below is the all-in-one PCA section using `merged_dic` and `stim_df`.


In [42]:
resolved_nwb = resolve_nwb_path(NWB_PATH)
mouse = nwb_loader(resolved_nwb)

df_trials = mouse.trials().reset_index(drop=True)
df_units = mouse.units().reset_index(drop=True)
stim_df = df_trials

print("NWB:", resolved_nwb)
print("trials shape:", df_trials.shape)
print("units shape:", df_units.shape)
print("trials columns:", list(df_trials.columns))
print("units columns:", list(df_units.columns))
stim_df

NWB: G:\Grant\neuropixels\nwb\Reach15
trials shape: (1352824, 4)
units shape: (4600, 9)
trials columns: ['start_time', 'stop_time', 'stimulus', 'optogenetics_LED_state']
units columns: ['depth', 'xpos', 'ypos', 'label', 'KSlabel', 'KSamplitude', 'KScontamination', 'probe', 'spike_times']


,start_time,stop_time,stimulus,optogenetics_LED_state
0,10239.422800,10239.422800,tone1_timestamps,0
1,10261.289967,10261.289967,tone1_timestamps,0
2,10283.329000,10283.329000,tone1_timestamps,0
3,10304.578900,10304.578900,tone1_timestamps,0
4,10382.254700,10382.254700,tone1_timestamps,0
...,...,...,...,...
1352819,18950.548600,18950.548600,opto_tagging_timestamps,1
1352820,18952.562467,18952.562467,opto_tagging_timestamps,1
1352821,18954.575767,18954.575767,opto_tagging_timestamps,1
1352822,18956.580033,18956.580033,opto_tagging_timestamps,1


### PCA Config


In [43]:
# Event selection for PCA
# Use raw event names from df_stim/stim_df (not block labels)
PCA_EVENT_TIME_COL = "event_time_s"
PCA_EVENT_LABEL_COL = "stimulus" if "stimulus" in stim_df.columns else ("label" if "label" in stim_df.columns else None)
PCA_EVENT_FILTER_COL = PCA_EVENT_LABEL_COL

# Keep only these event names (exact match). None => keep all (after exclusions).
PCA_EVENT_FILTER_VALUES = None   # example: ["reach_start", "pellet_contact"]

# Always exclude non-behavior frame timestamps from PCA unless you explicitly remove this entry.
PCA_EXCLUDE_EVENT_NAMES = ["frame_events_timestamps"]

# Probe/ROI selection
PCA_TARGET_PROBE = "A"
PCA_SELECTED_PROBES = None
PCA_ROI_FILTER = "IN_ROI"       # "IN_ROI", "OUT_ROI", or None

# Optional downsampling (disabled by default)
PCA_MAX_EVENTS = None
PCA_EVENT_SUBSAMPLE_MODE = "uniform"   # "uniform" or "first"

PCA_MAX_TENSOR_GB = 8.0




### PCA Helpers


In [44]:
def pca_select_events(stim_df,
                      event_time_col="event_time_s",
                      event_label_col=None,
                      event_filter_col=None,
                      event_filter_values=None,
                      exclude_event_names=None,
                      max_events=None,
                      subsample_mode="uniform"):
    if event_time_col not in stim_df.columns:
        raise ValueError(f"{event_time_col} not in stim_df columns: {list(stim_df.columns)}")

    out = stim_df.copy()
    out[event_time_col] = pd.to_numeric(out[event_time_col], errors="coerce")
    out = out.dropna(subset=[event_time_col]).sort_values(event_time_col).reset_index(drop=True)

    # Exclude nuisance/non-behavior event names (e.g. frame clocks)
    if exclude_event_names is not None and event_filter_col is not None and event_filter_col in out.columns:
        exclude_set = {str(x) for x in exclude_event_names}
        out = out[~out[event_filter_col].astype(str).isin(exclude_set)].reset_index(drop=True)

    # Keep only selected event names (exact match)
    if event_filter_values is not None and event_filter_col is not None and event_filter_col in out.columns:
        keep_set = {str(v) for v in event_filter_values}
        out = out[out[event_filter_col].astype(str).isin(keep_set)].reset_index(drop=True)

    # Optional subsampling (disabled by default)
    n_before = len(out)
    if max_events is not None and n_before > max_events:
        if subsample_mode == "first":
            idx = np.arange(max_events)
        else:
            idx = np.linspace(0, n_before - 1, max_events, dtype=int)
        out = out.iloc[idx].reset_index(drop=True)
        print(f"Downsampled events: {n_before} -> {len(out)} (mode={subsample_mode})")

    if len(out) == 0:
        raise ValueError("No events remain after filtering. Check PCA_EVENT_FILTER_VALUES/PCA_EXCLUDE_EVENT_NAMES.")

    if event_label_col is not None and event_label_col in out.columns:
        labels = out[event_label_col].astype(str).to_numpy()
    elif "stimulus" in out.columns:
        labels = out["stimulus"].astype(str).to_numpy()
    else:
        labels = np.array(["all_events"] * len(out), dtype=object)

    return out, out[event_time_col].to_numpy(dtype=float), labels


def pca_get_probe_units_df(merged_dic, probe, roi_filter=None):
    if probe not in merged_dic:
        raise ValueError(f"Probe {probe} not in merged_dic keys: {list(merged_dic.keys())}")

    df = merged_dic[probe].copy().reset_index(drop=True)
    if "spike_times" not in df.columns:
        raise ValueError(f"Probe {probe} DataFrame missing spike_times column.")

    if "probe" in df.columns:
        probe_norm = df["probe"].astype(str).str.strip().str.upper().str[0]
        df = df[probe_norm == str(probe).strip().upper()].reset_index(drop=True)

    if roi_filter is not None:
        if "in_brainRegion" not in df.columns:
            raise ValueError("ROI filter requested but in_brainRegion column is missing.")
        df = df[df["in_brainRegion"].astype(str) == str(roi_filter)].reset_index(drop=True)

    valid = df["spike_times"].apply(lambda x: isinstance(x, (list, np.ndarray))).to_numpy()
    df = df[valid].reset_index(drop=True)
    return df


def pca_bin_spikes_around_events(spike_times_list, event_times_s, win_start_s, win_end_s, bin_size_s, max_tensor_gb=8.0):
    edges = np.arange(win_start_s, win_end_s + bin_size_s, bin_size_s)
    n_bins = len(edges) - 1
    n_trials = len(event_times_s)
    n_units = len(spike_times_list)

    est_gb = (n_trials * n_units * n_bins * np.dtype(np.float32).itemsize) / (1024**3)
    print(f"Requested tensor shape=({n_trials}, {n_units}, {n_bins}) est_mem={est_gb:.2f} GB")
    if est_gb > max_tensor_gb:
        raise MemoryError(
            f"Estimated tensor memory {est_gb:.2f} GB exceeds PCA_MAX_TENSOR_GB={max_tensor_gb}. "
            "Reduce events/units or increase BIN_SIZE_S."
        )

    X = np.zeros((n_trials, n_units, n_bins), dtype=np.float32)
    for u, st in enumerate(spike_times_list):
        st = np.asarray(st, dtype=float)
        if st.size == 0:
            continue
        for t, t0 in enumerate(event_times_s):
            i0 = np.searchsorted(st, t0 + WINDOW_START_S, side="left")
            i1 = np.searchsorted(st, t0 + WINDOW_END_S, side="right")
            rel = st[i0:i1] - t0
            if rel.size:
                counts, _ = np.histogram(rel, bins=edges)
                X[t, u, :] = counts / bin_size_s

    return X, edges[:-1]


def pca_trial_level(trials, labels, n_components=12):
    X_trial = trials.mean(axis=2).T  # (units, trials)
    Xz = zscore_rows(X_trial)
    pca = PCA(n_components=min(n_components, Xz.shape[0], Xz.shape[1]))
    Xp = pca.fit_transform(Xz.T).T
    trial_types = pd.unique(labels)
    t_type_ind = [np.where(labels == t)[0] for t in trial_types]
    return Xp, pca.explained_variance_ratio_, trial_types, t_type_ind


def pca_trajectory(trials, labels, n_components=12):
    trial_types = pd.unique(labels)
    t_type_ind = [np.where(labels == t)[0] for t in trial_types]

    trial_averages = []
    kept_labels = []
    for t, idx in zip(trial_types, t_type_ind):
        if len(idx) > 0:
            trial_averages.append(trials[idx].mean(axis=0))
            kept_labels.append(t)

    if len(trial_averages) < 2:
        raise ValueError("Need at least 2 non-empty event labels for trajectory PCA.")

    Xa = np.hstack(trial_averages)
    Xaz = zscore_rows(Xa)
    pca = PCA(n_components=min(n_components, Xaz.shape[0], Xaz.shape[1]))
    Xa_p = pca.fit_transform(Xaz.T).T
    return Xa_p, pca.explained_variance_ratio_, kept_labels


def pca_plot_trial_level(Xp, trial_types, t_type_ind, title_prefix=""):
    projections = [(0, 1), (1, 2), (0, 2)]
    pal = sns.color_palette("colorblind", len(trial_types))

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (i, j) in zip(axes, projections):
        for k, t in enumerate(trial_types):
            idx = t_type_ind[k]
            ax.scatter(Xp[i, idx], Xp[j, idx], s=28, alpha=0.8, color=pal[k], label=str(t))
        ax.set_xlabel(f"PC {i+1}")
        ax.set_ylabel(f"PC {j+1}")
    axes[0].set_title(f"{title_prefix} Trial PCA")
    axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    sns.despine()
    plt.tight_layout()
    plt.show()


def pca_plot_trajectory(Xa_p, kept_labels, n_bins, time, smooth_sigma=2, title_prefix=""):
    pal = sns.color_palette("colorblind", len(kept_labels))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
    for comp in range(min(3, Xa_p.shape[0])):
        ax = axes[comp]
        for k, lbl in enumerate(kept_labels):
            s = k * n_bins
            e = (k + 1) * n_bins
            x = Xa_p[comp, s:e]
            if smooth_sigma and smooth_sigma > 0:
                x = gaussian_filter1d(x, sigma=smooth_sigma)
            ax.plot(time, x, lw=2, color=pal[k], label=str(lbl))
        ax.axvline(0, color="gray", ls="--", lw=1)
        ax.set_ylabel(f"PC {comp+1}")

    axes[1].set_xlabel("Time from event (s)")
    axes[0].set_title(f"{title_prefix} Trajectory PCA")
    axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    sns.despine()
    plt.tight_layout()
    plt.show()


### Ensure `stim_df` Exists for PCA Section
If `stim_df` is not already built, this creates it from `df_trials`.


In [45]:
df_trials

,start_time,stop_time,stimulus,optogenetics_LED_state
0,10239.422800,10239.422800,tone1_timestamps,0
1,10261.289967,10261.289967,tone1_timestamps,0
2,10283.329000,10283.329000,tone1_timestamps,0
3,10304.578900,10304.578900,tone1_timestamps,0
4,10382.254700,10382.254700,tone1_timestamps,0
...,...,...,...,...
1352819,18950.548600,18950.548600,opto_tagging_timestamps,1
1352820,18952.562467,18952.562467,opto_tagging_timestamps,1
1352821,18954.575767,18954.575767,opto_tagging_timestamps,1
1352822,18956.580033,18956.580033,opto_tagging_timestamps,1


In [46]:
if "stim_df" not in globals() or not isinstance(stim_df, pd.DataFrame) or stim_df.empty:
    if "df_trials" not in globals() or not isinstance(df_trials, pd.DataFrame) or df_trials.empty:
        raise ValueError("Need df_trials to build stim_df.")

    time_col = choose_event_time_col(df_trials)
    stim_df = df_trials.copy()
    stim_df["event_time_s"] = pd.to_numeric(stim_df[time_col], errors="coerce")
    stim_df = stim_df.dropna(subset=["event_time_s"]).sort_values("event_time_s").reset_index(drop=True)

    blk = build_block_labels(infer_is_opto(stim_df))
    stim_df = pd.concat([stim_df, blk], axis=1)
    stim_df["trial_index"] = np.arange(len(stim_df), dtype=int)
    stim_df["label"] = stim_df["stimulus"].astype(str) if "stimulus" in stim_df.columns else stim_df["block_label"].astype(str)

print("stim_df ready. columns:", list(stim_df.columns))
print("n_events:", len(stim_df))


stim_df ready. columns: ['start_time', 'stop_time', 'stimulus', 'optogenetics_LED_state']
n_events: 1352824


### Select Events for PCA


In [47]:
events_sel_df, pca_event_times, pca_event_labels = pca_select_events(
    stim_df=stim_df,
    event_time_col=PCA_EVENT_TIME_COL,
    event_label_col=PCA_EVENT_LABEL_COL,
    event_filter_col=PCA_EVENT_FILTER_COL,
    event_filter_values=PCA_EVENT_FILTER_VALUES,
    exclude_event_names=PCA_EXCLUDE_EVENT_NAMES,
    max_events=PCA_MAX_EVENTS,
    subsample_mode=PCA_EVENT_SUBSAMPLE_MODE,
)

print("Selected events:", len(events_sel_df))
if PCA_EVENT_FILTER_COL in events_sel_df.columns:
    print("\nSelected event counts:")
    print(events_sel_df[PCA_EVENT_FILTER_COL].astype(str).value_counts())
print("\nPCA label counts:")
print(pd.Series(pca_event_labels).value_counts())


ValueError: event_time_s not in stim_df columns: ['start_time', 'stop_time', 'stimulus', 'optogenetics_LED_state']

In [31]:
assert "frame_events_timestamps" not in events_sel_df[PCA_EVENT_FILTER_COL].astype(str).unique()
print("frame_events_timestamps excluded")


frame_events_timestamps excluded


### Single Probe PCA Run


In [32]:
probe_units_df = pca_get_probe_units_df(
    merged_dic=merged_dic,
    probe=PCA_TARGET_PROBE,
    roi_filter=PCA_ROI_FILTER,
)

spike_times_list = [np.asarray(x, dtype=float) for x in probe_units_df["spike_times"].values]

trials_probe, pca_time = pca_bin_spikes_around_events(
    spike_times_list=spike_times_list,
    event_times_s=pca_event_times,
    win_start_s=WINDOW_START_S,
    win_end_s=WINDOW_END_S,
    bin_size_s=BIN_SIZE_S,
    max_tensor_gb=PCA_MAX_TENSOR_GB,
)

print(f"Probe {PCA_TARGET_PROBE} | ROI={PCA_ROI_FILTER} | units={len(spike_times_list)} | trials={trials_probe.shape[0]}")

Xp, evr_trial, trial_types, t_type_ind = pca_trial_level(trials_probe, pca_event_labels, n_components=N_COMPONENTS)
print("Trial PCA EVR first 5:", np.round(evr_trial[:5], 4))
pca_plot_trial_level(Xp, trial_types, t_type_ind, title_prefix=f"Probe {PCA_TARGET_PROBE} ({PCA_ROI_FILTER})")

Xa_p, evr_traj, kept_labels = pca_trajectory(trials_probe, pca_event_labels, n_components=N_COMPONENTS)
print("Trajectory PCA EVR first 5:", np.round(evr_traj[:5], 4))
pca_plot_trajectory(
    Xa_p, kept_labels, n_bins=trials_probe.shape[2], time=pca_time,
    smooth_sigma=SMOOTH_SIGMA, title_prefix=f"Probe {PCA_TARGET_PROBE} ({PCA_ROI_FILTER})"
)


Requested tensor shape=(1352824, 554, 80) est_mem=223.36 GB


MemoryError: Estimated tensor memory 223.36 GB exceeds PCA_MAX_TENSOR_GB=8.0. Reduce events/units or increase BIN_SIZE_S.

### Multi-Probe PCA Runs
Runs selected/all probes and stores results in `pca_results_by_probe`.


In [ ]:
if PCA_SELECTED_PROBES is None:
    probes_to_run = sorted(list(merged_dic.keys()))
else:
    probes_to_run = [p for p in PCA_SELECTED_PROBES if p in merged_dic]

pca_results_by_probe = {}

for probe in probes_to_run:
    print("\n============================")
    print(f"Running probe {probe} | ROI={PCA_ROI_FILTER}")

    probe_units_df = pca_get_probe_units_df(merged_dic, probe=probe, roi_filter=PCA_ROI_FILTER)
    if probe_units_df.empty:
        print(f"Skipping probe {probe}: no units after filtering.")
        continue

    spike_times_list = [np.asarray(x, dtype=float) for x in probe_units_df["spike_times"].values]

    try:
        trials_probe, pca_time = pca_bin_spikes_around_events(
            spike_times_list=spike_times_list,
            event_times_s=pca_event_times,
            win_start_s=WINDOW_START_S,
            win_end_s=WINDOW_END_S,
            bin_size_s=BIN_SIZE_S,
            max_tensor_gb=PCA_MAX_TENSOR_GB,
        )
    except MemoryError as e:
        print(f"Skipping probe {probe} due to memory guard: {e}")
        continue

    Xp, evr_trial, trial_types, t_type_ind = pca_trial_level(trials_probe, pca_event_labels, n_components=N_COMPONENTS)
    Xa_p, evr_traj, kept_labels = pca_trajectory(trials_probe, pca_event_labels, n_components=N_COMPONENTS)

    pca_results_by_probe[probe] = {
        "units_df": probe_units_df,
        "trials": trials_probe,
        "time": pca_time,
        "trial_scores": Xp,
        "trial_evr": evr_trial,
        "trial_types": trial_types,
        "trial_type_ind": t_type_ind,
        "traj_scores": Xa_p,
        "traj_evr": evr_traj,
        "traj_labels": kept_labels,
    }

    print(f"Probe {probe}: units={trials_probe.shape[1]}, trials={trials_probe.shape[0]}")
    print("Trial EVR first 3:", np.round(evr_trial[:3], 4), "| Traj EVR first 3:", np.round(evr_traj[:3], 4))

print("\nCompleted probes:", list(pca_results_by_probe.keys()))


### Plot Any Probe from Multi-Probe Results


In [ ]:
PCA_PLOT_PROBE = PCA_TARGET_PROBE

if PCA_PLOT_PROBE not in pca_results_by_probe:
    raise ValueError(f"{PCA_PLOT_PROBE} not in pca_results_by_probe. Available: {list(pca_results_by_probe.keys())}")

res = pca_results_by_probe[PCA_PLOT_PROBE]
pca_plot_trial_level(
    Xp=res["trial_scores"],
    trial_types=res["trial_types"],
    t_type_ind=res["trial_type_ind"],
    title_prefix=f"Probe {PCA_PLOT_PROBE} ({PCA_ROI_FILTER})",
)
pca_plot_trajectory(
    Xa_p=res["traj_scores"],
    kept_labels=res["traj_labels"],
    n_bins=res["trials"].shape[2],
    time=res["time"],
    smooth_sigma=SMOOTH_SIGMA,
    title_prefix=f"Probe {PCA_PLOT_PROBE} ({PCA_ROI_FILTER})",
)


### Save PCA Summary


In [ ]:
summary_rows = []
for probe, res in pca_results_by_probe.items():
    summary_rows.append({
        "probe": probe,
        "roi_filter": PCA_ROI_FILTER,
        "n_units": int(res["trials"].shape[1]),
        "n_events": int(res["trials"].shape[0]),
        "trial_evr_pc1": float(res["trial_evr"][0]) if len(res["trial_evr"]) else np.nan,
        "traj_evr_pc1": float(res["traj_evr"][0]) if len(res["traj_evr"]) else np.nan,
    })

pca_summary_df = pd.DataFrame(summary_rows)
pca_summary_path = OUT_DIR / f"all_in_one_probe_roi_pca_{PCA_ROI_FILTER if PCA_ROI_FILTER is not None else 'ALLROI'}.csv"
pca_summary_df.to_csv(pca_summary_path, index=False)
print("Saved:", pca_summary_path)
pca_summary_df
